# RDF Extractions from all data

In [ ]:
import base64
import json
import os
from pathlib import Path
import requests
from PIL import Image

# 1. CONFIGURATION
# It's recommended to load the API key from an environment variable for security.
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

if GEMINI_API_KEY == "YOUR_API_KEY":
    raise ValueError("GEMINI_API_KEY not set. Please set your API key as an environment variable.")

MODEL = "gemini-2.5-flash"  # Using a more recent model, but you can change it back
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

# --- NEW: Specify your dataset directory ---
DATASET_DIR = "/content/drive/MyDrive/Research/Wageningen University /Datasets/Data"  # <--- IMPORTANT: Change this to the path of your dataset
OUTPUT_FILENAME = "rdf_extractions.json"


# 2. FUNCTIONS (Mostly unchanged, with minor improvements)

def encode_image_to_base64(image_path: str) -> str:
    """Encodes an image to a base64 string."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            buffer = Path(image_path).read_bytes()
            return base64.b64encode(buffer).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"An error occurred while processing the image at {image_path}: {e}")
        return None

def build_request(image_b64: str, prompt: str) -> dict:
    """Builds the JSON payload for the Gemini API request."""
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlineData": {"mimeType": "image/jpeg", "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def call_gemini(payload: dict) -> str:
    """Calls the Gemini API and returns the extracted RDF text."""
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        result = response.json()

        if "candidates" in result:
            extracted_rdf = result["candidates"][0]["content"]["parts"][0]["text"]
            # Clean up the response to get just the Turtle code
            if extracted_rdf.strip().startswith("```turtle"):
                extracted_rdf = extracted_rdf.strip()[len("```turtle"):-len("```")].strip()
            return extracted_rdf
        return None
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}\nResponse body: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# 3. MAIN EXECUTION (This is where the main changes are)

def main():
    """
    Main function to iterate through image datasets, extract RDF,
    and save the combined results.
    """
    # This is the same detailed prompt you provided.
    prompt_text = """
    You are a multimodal knowledge-extraction agent.



    \### Objective



    Extract concepts and relationships from diagrams, flowcharts, or tables and generate RDF triples in Turtle syntax using SKOS and a custom namespace.



    \### Instructions



    1\. \*\*Prefixes (always declare):\*\*



    &nbsp;  ```turtle

    &nbsp;  @prefix she:  <https://soilwise-he.github.io/soil-health#> .

    &nbsp;  @prefix skos: <http://www.w3.org/2004/02/skos/core#> .

    &nbsp;  ```



    2\. \*\*Nodes (Concepts):\*\*



    &nbsp;  \* Mint a URI in the `she:` namespace, using \*\*PascalCase\*\* (initial capitalization).

    &nbsp;  \* Declare as `a skos:Concept`.

    &nbsp;  \* Add `skos:prefLabel` with exact text from the image, using \*\*all lowercase\*\*.

    &nbsp;  \* Optionally add `skos:definition` if the image shows definitions.



    3\. \*\*Edges (Relationships):\*\*



    &nbsp;  \* Use `skos:narrower` or `skos:broader` for hierarchical links.

    &nbsp;  \* Otherwise, if it expresses a semantic relation that goes beyond SKOS, define a custom property in \*\*camelCase\*\* using clear natural language (e.g., `she:measures`, `she:affects`). Ensure your custom property name clearly reflects its meaning.

    &nbsp;  \* Custom properties must follow ontology property conventions and use the `she:` namespace.



    4\. \*\*Output Requirements:\*\*



    &nbsp;  \* Valid Turtle syntax only.

    &nbsp;  \* One triple per line, ending with a period.

    &nbsp;  \* Group all triples for the same subject with semicolons.

    &nbsp;  \* Do not use any other prefixes or ontologies besides `skos:` and `she:`.



    5\. Output \*\*only\*\* valid Turtle syntax:



    &nbsp;  \* One triple per line, ending with a period.

    &nbsp;  \* Group all triples for the same subject together, separated by semicolons.

    &nbsp;  \* Do not include any other prefixes or ontologies.



    \### Example Output Skeleton



    ```turtle

    @prefix she:  <https://soilwise-he.github.io/soil-health#> .

    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .



    she:TopConcept a skos:Concept ;

    &nbsp;   skos:prefLabel "top concept" ;

    &nbsp;   skos:narrower she:ChildConceptA,

    &nbsp;                 she:ChildConceptB ;

    &nbsp;   skos:definition "Top concept ..." .



    she:ChildConceptA a skos:Concept ;

    &nbsp;   skos:prefLabel "child concept a" ;

    &nbsp;   she:myCustomRelation she:OtherConcept .



    she:ChildConceptB a skos:Concept ;

    &nbsp;   skos:prefLabel "child concept b" .

    ```



    """

    all_rdf_data = []
    # --- NEW: Loop through the dataset directory ---
    for root, dirs, files in os.walk(DATASET_DIR):
        for file in files:
            # --- NEW: Check for image file extensions ---
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                print(f"Processing image: {img_path}")

                image_b64 = encode_image_to_base64(img_path)

                if image_b64:
                    request_body = build_request(image_b64, prompt_text)
                    print("Sending request to Gemini API...")
                    extracted_rdf = call_gemini(request_body)

                    if extracted_rdf:
                        print(f"Successfully extracted RDF from {img_path}")
                        # --- NEW: Append result to our list ---
                        all_rdf_data.append({
                            "source_image": img_path,
                            "rdf_graph_turtle": extracted_rdf
                        })
                    else:
                        print(f"Failed to get a valid response for {img_path}")
                print("-" * 20) # Separator for clarity

    # --- NEW: Save all collected RDF data to a single JSON file ---
    if all_rdf_data:
        output_data = {"dataset": all_rdf_data}
        with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4)
        print(f"\n--- Successfully saved all RDF extractions to {OUTPUT_FILENAME} ---")
    else:
        print("No RDF data was extracted from any images.")

if __name__ == "__main__":
    main()

Processing image: /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_10.2_cont.jpg
Sending request to Gemini API...
Successfully extracted RDF from /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_10.2_cont.jpg
--------------------
Processing image: /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_2.4.jpg
Sending request to Gemini API...
Successfully extracted RDF from /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_2.4.jpg
--------------------
Processing image: /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_7.3.jpg
Sending request to Gemini API...
Successfully extracted RDF from /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_7.3.jpg
--------------------
Processing image: /content/drive/MyDrive/Research/Wageningen University /Datasets/Data/tables/table_6.2.jpg
Sending request to Gemini AP